In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def convert_tz(df, cols, tz):
    out = df.copy()
    for col in cols:
        if out[col].dt.tz is not None:
            out[col] = out[col].dt.tz_convert(tz)
    return out

In [ ]:
data = pd.read_csv("biko-jobs.pseudonymised.csv", parse_dates=['creation', 'working', 'completed', 'cancelled'])
data = convert_tz(data, ['creation', 'working', 'completed', 'cancelled'], 'Europe/London')
data = data[data['complexMovement'] == False].drop(columns=['complexMovement'])

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

In [ ]:
data['working_duration'] = data.completed - data.working
data['time_in_queue'] = data.working - data.creation
data['total_lead_time'] = data.completed - data.creation

In [ ]:
data['session_date'] = data['creation'].dt.date
data['minutes_in_queue'] = data['time_in_queue'].dt.seconds/60

In [ ]:
creation_naive = (data.creation.dt.hour * 60 + data.creation.dt.minute)/60
creation_naive.hist()
plt.xticks(rotation=45)
plt.show()

In [ ]:
def numeric_bins(series, interval):
    min = np.floor((series / interval).min()) * interval
    max = np.ceil((series / interval).max()) * interval
    return np.arange(min, max + interval*0.1, interval)

In [ ]:
data['working_duration'].mean().seconds/60

In [ ]:
working_mins = data['working_duration'].dt.seconds / 60
working_mins.hist(bins=numeric_bins(working_mins, 10))

In [ ]:
plt.scatter((data['creation'].dt.hour * 60 + data['creation'].dt.minute)/60, data['total_lead_time'].dt.seconds/3600)
plt.xlabel('job creation')
plt.ylabel('total lead time')
plt.xticks(rotation=45);

In [ ]:
# Thanks Gemini
def count_active_at_creation_by_session(df, session_col="session_date"):
  """Calculates the number of active jobs in progress at the time each job

  is created, restricted strictly to jobs within the same session.
  """
  # Create a blank Series to hold the results mapped back to the original index
  results = pd.Series(index=df.index, dtype=int)

  # Group by session so each session is calculated independently
  for session_id, group in df.groupby(session_col):
    c = group["creation"].values.astype("datetime64[ns]")
    d = group["completed"].values.astype("datetime64[ns]")

    # Isolate valid (completed) completion times and sort them for this session
    nat_mask = np.isnat(d)
    d_valid = d[~nat_mask]
    d_valid.sort()

    # 1. Total jobs created on or before each job's creation time (in this session)
    started_before_or_at = np.searchsorted(c, c, side="right")

    # 2. Total jobs completed strictly before each job's creation time (in this session)
    completed_before = np.searchsorted(d_valid, c, side="left")

    # Active jobs = (started <= c) - (completed < c) - 1 (exclude current job)
    active = started_before_or_at - completed_before - 1

    # Store results back using the group's original DataFrame indices
    results.loc[group.index] = np.maximum(active, 0)

  return results


# Apply it to your DataFrame (replace 'session_date' with your actual session column name)
data["jobs_ahead"] = count_active_at_creation_by_session(
    data, session_col="session_date"
)

In [ ]:
plt.scatter(x=data['jobs_ahead'], y=data['total_lead_time'].dt.seconds/60)
plt.xlabel('queue size at time of triage')
plt.ylabel('total lead time (minutes)');

In [ ]:
cleaned = data.dropna(subset=['jobs_ahead', 'time_in_queue'])
slope, intercept = np.polyfit(cleaned['jobs_ahead'], cleaned['time_in_queue'].dt.seconds/60, 1)
slope, intercept

In [ ]:
sns.scatterplot(data=data, x='jobs_ahead', y='minutes_in_queue', hue='session_date')

In [ ]:
data['working_duration_minutes'] = data['working_duration'].dt.total_seconds()/60

In [ ]:
data['total_lead_time_minutes'] = data['total_lead_time'].dt.total_seconds()/60
sns.scatterplot(data=data, x='jobs_ahead', y='total_lead_time_minutes', hue='session_date')

In [ ]:
plt.scatter(x=data['session_date'], y=data['total_lead_time_minutes'])
plt.ylabel('Total lead time (minutes)')
plt.xticks(rotation=45);

In [ ]:
plt.scatter(x=data['session_date'], y=data['working_duration_minutes'])
plt.ylabel('Working duration (minutes)')
plt.xticks(rotation=45);